# Week 1 - Day 3 Lab: Data & Matrix Manipulation
In this lab, you'll work with a realistic weather dataset. You'll use **Pandas** to explore and clean the data, and **NumPy** to perform matrix operations.

**Dataset:** `hourly_weather_10_days.csv` (10 days of hourly weather data)

## Step 1: Load the Data
- Use Pandas to load the CSV file
- Display the first few rows
- Check the number of rows and columns

In [1]:
# Load the data into a DataFrame
import pandas as pd

# Replace the file path if needed
df = pd.read_csv('hourly_weather_10_days.csv')
print(f"Shape of dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

Shape of dataset: 240 rows, 6 columns


,timestamp,temperature_C,humidity_%,wind_speed_kmph,pressure_hPa,visibility_km
0,2023-03-01 00:00:00,16.6,74.4,5.7,1012.5,9.5
1,2023-03-01 01:00:00,16.2,78.5,5.0,1012.1,10.3
2,2023-03-01 02:00:00,15.3,73.3,4.7,NaN,11.1
3,2023-03-01 03:00:00,15.8,72.4,1.3,1005.0,8.9
4,2023-03-01 04:00:00,20.9,70.6,6.8,1016.3,9.8


## Step 2: Basic Exploration
- Check column names and data types
- Display basic statistics using `.describe()`
- Count missing values in each column

In [2]:
# Explore the DataFrame
print("--- Column info & data types ---")
print(df.info())

print("\n--- Summary statistics ---")
print(df.describe())

print("\n--- Missing values per column ---")
print(df.isna().sum())

--- Column info & data types ---
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   timestamp        240 non-null    str    
 1   temperature_C    228 non-null    float64
 2   humidity_%       224 non-null    float64
 3   wind_speed_kmph  226 non-null    float64
 4   pressure_hPa     223 non-null    float64
 5   visibility_km    228 non-null    float64
dtypes: float64(5), str(1)
memory usage: 11.4 KB
None

--- Summary statistics ---
       temperature_C  humidity_%  wind_speed_kmph  pressure_hPa  visibility_km
count     228.000000  224.000000       226.000000    223.000000     228.000000
mean       21.315789   66.795982        10.105310   1011.884753       9.989474
std         3.421237    8.190300         3.940668      5.187080       1.022166
min        11.500000   47.800000         1.300000    998.100000       6.800000
25%        18.700000   61.075

## Step 3: Handle Missing Values
- Drop or fill missing values
- Justify your approach (e.g., fill with mean, forward fill, etc.)

In [3]:
# Fill missing values
# Approach: All missing values are in numeric weather metrics (temperature,
# humidity, wind speed, pressure, visibility). Since these are continuous
# hourly readings that don't change drastically hour to hour, the best
# approach is to fill gaps using linear interpolation (based on neighboring
# hours) rather than a single global mean, which would ignore day/night
# and weather trends. Any remaining edge NaNs (start/end of series) are
# then filled with the column mean as a fallback.

numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    df[col] = df[col].interpolate(method='linear')   # fill using nearby hours
    df[col] = df[col].fillna(df[col].mean())          # fallback for edge NaNs

print("Missing values after cleaning:")
print(df.isna().sum())

Missing values after cleaning:
timestamp          0
temperature_C      0
humidity_%         0
wind_speed_kmph    0
pressure_hPa       0
visibility_km      0
dtype: int64


## Step 4: Data Analysis
- Calculate daily average temperature
- Find max, min, mean for each metric
- Which hour of the day is the most humid on average?

In [4]:
# Perform analysis
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour

# Daily average temperature
daily_avg_temp = df.groupby('date')['temperature_C'].mean()
print("--- Daily average temperature (°C) ---")
print(daily_avg_temp)

# Max, min, mean for each metric (overall)
metrics = ['temperature_C', 'humidity_%', 'wind_speed_kmph', 'pressure_hPa', 'visibility_km']
summary_stats = df[metrics].agg(['max', 'min', 'mean'])
print("\n--- Max / Min / Mean for each metric ---")
print(summary_stats)

# Which hour of the day is most humid on average?
avg_humidity_by_hour = df.groupby('hour')['humidity_%'].mean()
most_humid_hour = avg_humidity_by_hour.idxmax()
print("\n--- Average humidity by hour ---")
print(avg_humidity_by_hour)
print(f"\nMost humid hour of the day: {most_humid_hour}:00 "
      f"(avg humidity = {avg_humidity_by_hour.max():.2f}%)")

--- Daily average temperature (°C) ---
date
2023-03-01    21.495833
2023-03-02    21.372917
2023-03-03    21.410417
2023-03-04    21.581250
2023-03-05    21.641667
2023-03-06    21.858333
2023-03-07    21.077083
2023-03-08    20.916667
2023-03-09    20.918750
2023-03-10    21.697917
Name: temperature_C, dtype: float64

--- Max / Min / Mean for each metric ---
      temperature_C  humidity_%  wind_speed_kmph  pressure_hPa  visibility_km
max       28.700000      88.100         17.80000    1027.00000         12.600
min       11.500000      47.800          1.30000     998.10000          6.800
mean      21.397083      66.905         10.06375    1011.86375          9.995

--- Average humidity by hour ---
hour
0     78.170
1     78.420
2     75.735
3     71.940
4     69.310
5     69.455
6     65.770
7     64.810
8     63.490
9     59.650
10    58.710
11    58.910
12    59.010
13    58.330
14    60.850
15    61.145
16    59.300
17    64.030
18    67.210
19    69.190
20    67.670
21    72.185
2

## Step 5: NumPy Matrix Exercises
Convert relevant DataFrame columns into NumPy arrays and perform matrix operations.

In [5]:
# Extract temperature and wind_speed as NumPy arrays
import numpy as np

temp = df['temperature_C'].to_numpy()
wind = df['wind_speed_kmph'].to_numpy()

print(f"temp array shape: {temp.shape}")
print(f"wind array shape: {wind.shape}")

temp array shape: (240,)
wind array shape: (240,)


### a) Reshape into matrix form
- Assume each row is a day
- Reshape temperature into a (10, 24) matrix
- Calculate daily min, max, and mean using axis-based operations

In [6]:
# Reshape and aggregate
# Each row = one day (24 hourly readings per day, 10 days total)
temp_matrix = temp.reshape((10, 24))
print("Temperature matrix shape:", temp_matrix.shape)

daily_min = temp_matrix.min(axis=1)
daily_max = temp_matrix.max(axis=1)
daily_mean = temp_matrix.mean(axis=1)

for day in range(temp_matrix.shape[0]):
    print(f"Day {day + 1}: min={daily_min[day]:.2f}°C, "
          f"max={daily_max[day]:.2f}°C, mean={daily_mean[day]:.2f}°C")

Temperature matrix shape: (10, 24)
Day 1: min=14.70°C, max=28.20°C, mean=21.50°C
Day 2: min=15.70°C, max=28.70°C, mean=21.37°C
Day 3: min=13.60°C, max=25.70°C, mean=21.41°C
Day 4: min=15.90°C, max=27.10°C, mean=21.58°C
Day 5: min=12.40°C, max=24.90°C, mean=21.64°C
Day 6: min=15.50°C, max=26.20°C, mean=21.86°C
Day 7: min=15.30°C, max=25.90°C, mean=21.08°C
Day 8: min=13.50°C, max=26.00°C, mean=20.92°C
Day 9: min=14.30°C, max=27.10°C, mean=20.92°C
Day 10: min=11.50°C, max=28.50°C, mean=21.70°C


### b) Normalize the temperature matrix
- Subtract the mean and divide by std deviation
- Do it manually using NumPy functions

In [7]:
# Normalize temp_matrix
def normalize(matrix):
    """Standardize a matrix: subtract mean and divide by std deviation."""
    mean = matrix.mean()
    std = matrix.std()
    return (matrix - mean) / std

normalized_temp_matrix = normalize(temp_matrix)

print("Original matrix mean: {:.4f}, std: {:.4f}".format(
    temp_matrix.mean(), temp_matrix.std()))
print("Normalized matrix mean: {:.4f}, std: {:.4f}".format(
    normalized_temp_matrix.mean(), normalized_temp_matrix.std()))
print("\nNormalized matrix (first day):")
print(normalized_temp_matrix[0])

Original matrix mean: 21.3971, std: 3.4128
Normalized matrix mean: -0.0000, std: 1.0000

Normalized matrix (first day):
[-1.40561789 -1.52282393 -1.78653753 -1.64002998 -0.14565293 -0.17495444
  0.41107578  0.32317125 -0.05774839  1.99335736  1.61243772  1.23151808
  1.17291506  0.58688484  1.8175483   0.70409089  0.17666369  0.67478938
  0.49898031  0.2059652  -0.93679372 -0.58517559 -0.49727106 -1.9623466 ]


### c) Apply custom mask/filter
- Create a mask for wind speed > 15 kmph
- Use it to extract high-wind readings

In [8]:
# Create boolean mask and filter wind speeds
mask = wind > 15
high_wind = wind[mask]

print(f"Number of high-wind readings (>15 kmph): {high_wind.size}")
print("High wind readings:")
print(high_wind)

Number of high-wind readings (>15 kmph): 30
High wind readings:
[17.6 16.  16.5 16.3 16.7 15.8 17.8 15.1 16.3 15.2 17.  15.9 15.6 15.8
 15.4 15.6 16.3 15.3 16.2 16.9 15.3 15.2 15.5 17.4 17.4 15.4 15.4 16.5
 17.  15.7]


## Final Challenge: Write Your Own Function
Write a function `daily_summary(matrix)` that takes a NumPy matrix of shape (10, 24) and returns a summary dictionary for each day.

In [9]:
# Write and test the function
def daily_summary(matrix):
    """
    Takes a NumPy matrix of shape (10, 24) representing 10 days of
    24 hourly readings, and returns a list of dictionaries with the
    min, max, and mean for each day.
    """
    summaries = []
    for day_index in range(matrix.shape[0]):
        day_data = matrix[day_index]
        summaries.append({
            "day": day_index + 1,
            "min": float(day_data.min()),
            "max": float(day_data.max()),
            "mean": float(day_data.mean())
        })
    return summaries

# Example usage:
summaries = daily_summary(temp_matrix)
for s in summaries:
    print(f"Day {s['day']}: min={s['min']:.2f}, max={s['max']:.2f}, mean={s['mean']:.2f}")

Day 1: min=14.70, max=28.20, mean=21.50
Day 2: min=15.70, max=28.70, mean=21.37
Day 3: min=13.60, max=25.70, mean=21.41
Day 4: min=15.90, max=27.10, mean=21.58
Day 5: min=12.40, max=24.90, mean=21.64
Day 6: min=15.50, max=26.20, mean=21.86
Day 7: min=15.30, max=25.90, mean=21.08
Day 8: min=13.50, max=26.00, mean=20.92
Day 9: min=14.30, max=27.10, mean=20.92
Day 10: min=11.50, max=28.50, mean=21.70


## ✅ Submit your notebook once complete.
- Add comments where necessary